In [36]:
# DSBDA Assignment 4-1 - Visualization on Air Quality Dataset

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [37]:
# Load dataset
df = pd.read_csv('AirQuality.csv', delimiter=';')

In [38]:
df.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
0,10/03/2004,18.00.00,"2,6",1360.0,150.0,"11,9",1046.0,166.0,1056.0,113.0,1692.0,1268.0,"13,6","48,9","0,7578",NaN,NaN
1,10/03/2004,19.00.00,2,1292.0,112.0,"9,4",955.0,103.0,1174.0,92.0,1559.0,972.0,"13,3","47,7","0,7255",NaN,NaN
2,10/03/2004,20.00.00,"2,2",1402.0,88.0,"9,0",939.0,131.0,1140.0,114.0,1555.0,1074.0,"11,9","54,0","0,7502",NaN,NaN
3,10/03/2004,21.00.00,"2,2",1376.0,80.0,"9,2",948.0,172.0,1092.0,122.0,1584.0,1203.0,"11,0","60,0","0,7867",NaN,NaN
4,10/03/2004,22.00.00,"1,6",1272.0,51.0,"6,5",836.0,131.0,1205.0,116.0,1490.0,1110.0,"11,2","59,6","0,7888",NaN,NaN


In [39]:
df.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
0,10/03/2004,18.00.00,"2,6",1360.0,150.0,"11,9",1046.0,166.0,1056.0,113.0,1692.0,1268.0,"13,6","48,9","0,7578",NaN,NaN
1,10/03/2004,19.00.00,2,1292.0,112.0,"9,4",955.0,103.0,1174.0,92.0,1559.0,972.0,"13,3","47,7","0,7255",NaN,NaN
2,10/03/2004,20.00.00,"2,2",1402.0,88.0,"9,0",939.0,131.0,1140.0,114.0,1555.0,1074.0,"11,9","54,0","0,7502",NaN,NaN
3,10/03/2004,21.00.00,"2,2",1376.0,80.0,"9,2",948.0,172.0,1092.0,122.0,1584.0,1203.0,"11,0","60,0","0,7867",NaN,NaN
4,10/03/2004,22.00.00,"1,6",1272.0,51.0,"6,5",836.0,131.0,1205.0,116.0,1490.0,1110.0,"11,2","59,6","0,7888",NaN,NaN


### a. Data Cleaning
In this step, we will:
- Drop irrelevant/unnamed columns.
- Fix data types by replacing European decimal commas with dots and casting to float.
- Remove duplicate rows.
- Handle missing values (NaNs) by filling numeric columns with their mean and dropping any remaining NaNs.

In [40]:
# Rename columns for clarity
df = df.rename(columns={'T': 'Temperature', 'RH': 'Relative Humidity', 'AH': 'Absolute Humidity'})

# 1. Drop unnamed columns if present
df = df.drop(['Unnamed: 15', 'Unnamed: 16'], axis=1, errors='ignore')

# 2. Fix data types (replace commas with dots and convert to float)
cols_to_float = ['CO(GT)', 'C6H6(GT)', 'Temperature', 'Relative Humidity', 'Absolute Humidity']
for col in cols_to_float:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(',', '.').astype(float)

# 3. Remove duplicate rows
df = df.drop_duplicates()

# 4. Fill numeric NaNs with mean and drop remaining
df = df.fillna(df.mean(numeric_only=True))
df = df.dropna()

display(df.head())

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),Temperature,Relative Humidity,Absolute Humidity
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888


### d. Error Correcting
In this step, we systematically identify and correct erroneous data points (outliers) using the Interquartile Range (IQR) method.

In [43]:
# Function to remove outliers using IQR
def remove_outliers(column):
    Q1 = column.quantile(0.25)
    Q3 = column.quantile(0.75)
    IQR = Q3 - Q1
    threshold = 1.5 * IQR
    outlier_mask = (column < Q1 - threshold) | (column > Q3 + threshold)
    return column[~outlier_mask]

# Apply outlier removal
cols_for_outliers = ['Temperature', 'Relative Humidity', 'Absolute Humidity',
                     'PT08.S4(NO2)', 'PT08.S5(O3)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'PT08.S1(CO)']
for col in cols_for_outliers:
    if col in df.columns:
        df[col] = remove_outliers(df[col])

print('Data processing complete. Final dataset shape:', df.shape)

Data processing complete. Final dataset shape: (9357, 19)


### b. Data Integration
In this step, we integrate temporal features by parsing the `Date` column into separate `Year` and `Month` dimensions, which can later be joined or aggregated across different granularities.

In [41]:
# Convert Date to datetime
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')

# Extract new features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['yearr'] = df['Year'].astype(str)
df['month'] = df['Month'].astype(str)

display(df[['Date', 'Year', 'Month']].head())

,Date,Year,Month
0,2004-03-10,2004,3
1,2004-03-10,2004,3
2,2004-03-10,2004,3
3,2004-03-10,2004,3
4,2004-03-10,2004,3


### c. Data Transformation
In this step, we apply mathematical transformations to normalize or scale variables. Here we scale `Absolute Humidity`.

In [42]:
# Scale Absolute Humidity
df['Absolute Humidity'] = df['Absolute Humidity'].multiply(100)

display(df[['Absolute Humidity']].head())

,Absolute Humidity
0,75.78
1,72.55
2,75.02
3,78.67
4,78.88
